# ЛР 5



In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Загрузка и предобработка (Ваш код)
DATA_PATH = "person_2025_update.csv.bz2"

df = pd.read_csv(DATA_PATH, low_memory=False, compression='bz2')

for col in ['birthyear','deathyear','hpi','hpi_raw','age','non_en_page_views','prob_ratio','coefficient_of_variation','l','l_']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for date_col in ['birthdate','deathdate']:
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

if 'birthyear' in df.columns:
    df['birth_decade'] = (df['birthyear']//10)*10
if 'birthyear' in df.columns and 'deathyear' in df.columns:
    df['computed_age'] = df['deathyear'] - df['birthyear']
if 'gender' in df.columns:
    df['gender'] = df['gender'].fillna('unknown').astype(str)
if 'occupation' in df.columns:
    df['occupation'] = df['occupation'].fillna('UNKNOWN').astype(str)

print("Данные загружены и подготовлены.")

###Гистограммы годов рождения (Graph Objects + Dropdown/Buttons)


In [ ]:
# Подготовка данных
by = df['birthyear'].dropna()
pre1800 = by[by < 1800]
post1800 = by[by >= 1800]

# Создаем объект Figure
fig = go.Figure()

# Добавляем трасс (Trace) для данных до 1800 года
fig.add_trace(go.Histogram(
    x=pre1800,
    xbins=dict(size=100), # шаг 100 лет
    marker_color='skyblue',
    name='Pre-1800',
    visible=True # По умолчанию виден этот график
))

# Добавляем трасс для данных после 1800 года
fig.add_trace(go.Histogram(
    x=post1800,
    xbins=dict(size=10), # шаг 10 лет
    marker_color='salmon',
    name='Post-1800',
    visible=False # По умолчанию скрыт
))

# Создаем меню (updatemenus) для переключения
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            buttons=list([
                dict(
                    args=[{"visible": [True, False]},
                          {"title": "Birth years before 1800 (100-year bins)", "xaxis.range": [pre1800.min(), 1800]}],
                    label="Pre-1800",
                    method="update"
                ),
                dict(
                    args=[{"visible": [False, True]},
                          {"title": "Birth years from 1800 onwards (10-year bins)", "xaxis.range": [1800, post1800.max()]}],
                    label="Post-1800",
                    method="update"
                )
            ]),
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.5,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
    ],
    title="Birth years distribution (Select period above)",
    xaxis_title="Year of birth",
    yaxis_title="Count"
)

fig.show()

### Топ-10 профессий (Plotly Express Bar)


In [ ]:
top_occupations = df['occupation'].value_counts().nlargest(10).reset_index()
top_occupations.columns = ['Occupation', 'Count']

fig = px.bar(
    top_occupations,
    x='Count',
    y='Occupation',
    orientation='h',
    title='Top 10 Occupations in Dataset',
    text='Count', # Добавляем подписи значений
    color='Count', # Раскрашиваем для красоты
    color_continuous_scale='Viridis'
)

# Переворачиваем ось Y, чтобы топ-1 был сверху
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

### Распределение по полу (Plotly Express Pie)

In [ ]:
g = df['gender'].fillna('unknown').value_counts().reset_index()
g.columns = ['Gender', 'Count']

fig = px.pie(
    g,
    values='Count',
    names='Gender',
    title='Gender Distribution',
    hole=0.4 # Делаем график "пончиком"
)

fig.update_traces(textinfo='percent+label')
fig.show()

###Топ-20 стран рождения (Plotly Express Line)


In [ ]:
country_counts = df['bplace_country'].value_counts().head(20).reset_index()
country_counts.columns = ['Country', 'Count']

fig = px.line(
    country_counts,
    x='Country',
    y='Count',
    title='Number of persons by birth country (Top 20)',
    markers=True, # Добавляем точки
)

fig.update_traces(line_color='green')
fig.show()

###Strip Plot (Non-English views by Gender)


In [ ]:
sub = df[df['non_en_page_views'].notna() & df['gender'].notna()]

fig = px.strip(
    sub,
    x='gender',
    y='non_en_page_views',
    color='gender', # Разные цвета для полов
    title='Distribution of non-English page views by gender',
    log_y=True, # Логарифмическая шкала включается одним параметром
    stripmode='overlay'
)

fig.update_layout(showlegend=False)
fig.show()

###HPI vs Page Views с АНИМАЦИЕЙ

In [ ]:
# Подготовка данных для анимации (убираем пустые значения)
sub_anim = df[
    df['hpi'].notna() &
    df['non_en_page_views'].notna() &
    df['birth_decade'].notna()
].copy()

sub_anim = sub_anim[(sub_anim['birth_decade'] >= 1800) & (sub_anim['birth_decade'] <= 2000)]
sub_anim = sub_anim.sort_values('birth_decade')

fig = px.scatter(
    sub_anim,
    x='non_en_page_views',
    y='hpi',
    animation_frame='birth_decade', # Ключевой параметр для анимации
    animation_group='hpi', # Группировка
    log_x=True, # Логарифмическая ось X (вместо ручного np.log10)
    color='gender', # Дополнительное измерение цветом
    hover_name='occupation', # При наведении покажет профессию
    title='HPI vs Page Views (Animated by Birth Decade)',
    range_x=[1, sub_anim['non_en_page_views'].max()], # Фиксируем оси, чтобы график не скакал
    range_y=[sub_anim['hpi'].min(), sub_anim['hpi'].max()]
)

fig.show()

###Распределение рождений по месяцам (Bar)

In [ ]:
bd = pd.to_datetime(df['birthdate'], errors='coerce').dropna()
month_counts = bd.dt.month.value_counts().sort_index().reset_index()
month_counts.columns = ['Month_Num', 'Count']

# Словарь для маппинга номеров в названия
month_map = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
             7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
month_counts['Month_Name'] = month_counts['Month_Num'].map(month_map)

fig = px.bar(
    month_counts,
    x='Month_Name',
    y='Count',
    title='Number of Persons Born by Month',
    color='Count',
    # В Plotly используем 'RdBu_r' вместо 'coolwarm'
    color_continuous_scale='RdBu_r'
)

fig.show()

### Плотность распределения HPI (Histogram/KDE approximation)

In [ ]:
fig = px.histogram(
    df,
    x="hpi",
    nbins=50,
    marginal="violin", # Добавляет скрипичный график (или 'rug', 'box') сбоку
    title='Distribution of HPI',
    opacity=0.7,
    color_discrete_sequence=['indianred']
)
fig.show()

### Матрица корреляции (Heatmap)

In [ ]:
num_cols = ['hpi','hpi_raw','birthyear','deathyear','age','computed_age','non_en_page_views','prob_ratio','coefficient_of_variation']
exist = [c for c in num_cols if c in df.columns]
corr_matrix = df[exist].corr()

fig = px.imshow(
    corr_matrix,
    text_auto='.2f', # Автоматически показывать значения с точностью 2 знака
    aspect="auto",
    title='Correlation matrix of numeric features',
    color_continuous_scale='RdBu_r',
    origin='lower'
)
fig.show()

### Гео-данные (Scatter Geo)

In [ ]:
sub_geo = df[df['bplace_lon'].notna() & df['bplace_lat'].notna()]

fig = px.scatter_geo(
    sub_geo,
    lon='bplace_lon',
    lat='bplace_lat',
    color='hpi', # Цвет зависит от HPI
    hover_name='occupation', # При наведении покажет профессию
    size_max=15,
    title='Birthplace coordinates colored by HPI',
    projection="natural earth", # Красивая проекция карты
    color_continuous_scale='viridis'
)

fig.show()